In [13]:
# ============================================================
# TRANSFORM RACES DATA — LOCAL EXECUTION
# ============================================================

import os
from pyspark.sql import functions as F
from pyspark.sql.types import *
import nbformat
from nbconvert import PythonExporter

In [14]:
# -----------------------------------------
# 1. Load environment + helpers
# -----------------------------------------
def run_notebook(path):
    with open(path, "r") as f:
        nb = nbformat.read(f, as_version=4)
    source, _ = PythonExporter().from_notebook_node(nb)
    exec(source, globals())

run_notebook("/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")
run_notebook("/Users/manoharazalki/F1-Analytics/notebooks/00-common/03_silver_helpers.ipynb")

===== F1 Analytics Environment Configuration =====
BASE_PATH:        /Users/manoharazalki/F1-Analytics/data
LANDING_PATH:     /Users/manoharazalki/F1-Analytics/data/landing
BRONZE_PATH:      /Users/manoharazalki/F1-Analytics/data/bronze
SILVER_PATH:      /Users/manoharazalki/F1-Analytics/data/silver
GOLD_PATH:        /Users/manoharazalki/F1-Analytics/data/gold
INCREMENTAL_PATH: /Users/manoharazalki/F1-Analytics/data/incremental
CONTROL_PATH:     /Users/manoharazalki/F1-Analytics/data/control


In [15]:
# -----------------------------------------
# 2. Receive p_batch_id (unified logic)
# -----------------------------------------

# Case 1: Papermill parameter
if "p_batch_id" in globals() and p_batch_id not in [None, ""]:
    pass

# Case 2: Databricks widget
elif "dbutils" in globals():
    p_batch_id = dbutils.widgets.get("p_batch_id")

# Case 3: Manual Jupyter execution → auto-detect latest batch
else:
    bronze_folder = f"{BRONZE_PATH}/races/data.parquet"
    batch_folders = [
        f.split("=")[1]
        for f in os.listdir(bronze_folder)
        if f.startswith("batch_id=")
    ]
    if not batch_folders:
        raise Exception("❌ No batch_id partitions found in Bronze races")
    p_batch_id = sorted(batch_folders)[-1]
    print("⚠️ Auto-selected latest batch_id:", p_batch_id)

# Final validation
if p_batch_id is None or p_batch_id == "":
    raise Exception("❌ p_batch_id not provided to Silver notebook")

print("Using p_batch_id:", p_batch_id)

Using p_batch_id: 20260609_131145


In [16]:
# -----------------------------------------
# 3. Define Bronze + Silver paths
# -----------------------------------------
bronze_path = f"{BRONZE_PATH}/races/data.parquet"
silver_path = f"{SILVER_PATH}/races"
os.makedirs(silver_path, exist_ok=True)

In [17]:
# -----------------------------------------
# 4. Read Bronze (ONLY this batch)
# -----------------------------------------
read_df = (
    spark.read.parquet(bronze_path)
    .filter(F.col("batch_id") == p_batch_id)
)

print("✔ Bronze races read")
read_df.show(5)

✔ Bronze races read
+------+-----+--------------------+------------------+----------+------------+--------------------+--------------------+---------------+
|season|round|                 url|          raceName|      date|   circuitId|    ingest_timestamp|         source_file|       batch_id|
+------+-----+--------------------+------------------+----------+------------+--------------------+--------------------+---------------+
|  1950|    1|https://en.wikipe...|british grand prix|1950-05-13| silverstone|2026-06-09 13:11:...|/Users/manoharaza...|20260609_131145|
|  1950|    2|https://en.wikipe...| monaco grand prix|1950-05-21|      monaco|2026-06-09 13:11:...|/Users/manoharaza...|20260609_131145|
|  1950|    3|https://en.wikipe...|  indianapolis 500|1950-05-30|indianapolis|2026-06-09 13:11:...|/Users/manoharaza...|20260609_131145|
|  1950|    4|https://en.wikipe...|  swiss grand prix|1950-06-04|  bremgarten|2026-06-09 13:11:...|/Users/manoharaza...|20260609_131145|
|  1950|    5|https:/

In [18]:
# -----------------------------------------
# 5. Select required columns
# -----------------------------------------
races_selected_df = read_df.select(
    "season",
    "round",
    "circuitId",
    "date",
    "raceName",
    "ingest_timestamp",
    "source_file",
    "batch_id"
)

In [19]:
# -----------------------------------------
# 6. Standardize + rename columns
# -----------------------------------------
races_renamed_df = (
    races_selected_df
        .withColumnRenamed("season", "year")
        .withColumnRenamed("circuitId", "circuit_id")
        .withColumnRenamed("date", "race_date")
        .withColumnRenamed("raceName", "race_name")
)

In [20]:
# -----------------------------------------
# 7. Remove duplicates
# -----------------------------------------
races_distinct_df = races_renamed_df.dropDuplicates(["year", "round"])

In [21]:
# -----------------------------------------
# 8. Title Case transformation
# -----------------------------------------
races_final_df = (
    races_distinct_df
        .withColumn("race_name", F.initcap("race_name"))
)

In [22]:
# -----------------------------------------
# 9. Write to Silver
# -----------------------------------------
write_to_silver(
    races_final_df,
    f"{silver_path}/data.parquet",
    merge_keys=["year", "round"]
)

print("✔ Races Silver written:", f"{silver_path}/data.parquet")

✔ Races Silver written: /Users/manoharazalki/F1-Analytics/data/silver/races/data.parquet


In [23]:
# -----------------------------------------
# 10. Validate
# -----------------------------------------
spark.read.parquet(f"{silver_path}/data.parquet").show(5)

+----+-----+------------+----------+------------------+--------------------+--------------------+---------------+--------------------+--------------------+
|year|round|  circuit_id| race_date|         race_name|    ingest_timestamp|         source_file|       batch_id|   created_timestamp|   updated_timestamp|
+----+-----+------------+----------+------------------+--------------------+--------------------+---------------+--------------------+--------------------+
|1950|    1| silverstone|1950-05-13|British Grand Prix|2026-06-09 13:11:...|/Users/manoharaza...|20260609_131145|2026-06-09 13:41:...|2026-06-09 13:41:...|
|1950|    2|      monaco|1950-05-21| Monaco Grand Prix|2026-06-09 13:11:...|/Users/manoharaza...|20260609_131145|2026-06-09 13:41:...|2026-06-09 13:41:...|
|1950|    3|indianapolis|1950-05-30|  Indianapolis 500|2026-06-09 13:11:...|/Users/manoharaza...|20260609_131145|2026-06-09 13:41:...|2026-06-09 13:41:...|
|1950|    4|  bremgarten|1950-06-04|  Swiss Grand Prix|2026-06-0